# 1.5 — Discrete / Masked Diffusion

**Mechanism of the day:** diffusion where the "noise" is a `[MASK]` token. Corrupt a
sequence by masking a *random fraction* of it, learn to undo that, then generate by
unmasking a few tokens at a time.

This rung closes the loop between the last three notebooks, and it is the core of
**DPLM** and the modern discrete generative protein models.

Recall where we left things:

- **1.2** trained a masked LM at a *fixed* 15% rate, then generated by Gibbs
  sampling. It worked, but we had to admit it was a heuristic: the learned
  conditionals need not agree with any joint distribution, so the chain had no
  guaranteed target, and it cost `4 sweeps x 28 positions = 112` network
  evaluations.
- **1.3 / 1.4** built proper generative processes for *continuous* data by defining a
  forward corruption and learning its reverse.

Masked diffusion is what happens when you apply the 1.3 recipe to the 1.2 objective.
Define a forward process that masks each token independently with probability `t`:

```
t = 0   ->  the clean sequence
t = 1   ->  every token is [MASK]
```

This is an **absorbing-state** diffusion: `[MASK]` is a state you fall into and never
leave. The reverse process therefore only ever *unmasks*, which is a much easier job
than general denoising, and it gives a genuine ELBO on the likelihood.

**The change from 1.2 is almost embarrassingly small** — mask at a rate drawn from
`U(0, 1)` rather than fixed at 0.15. That is nearly the whole difference. But it buys
two things:

1. The model becomes competent at **every** corruption level, not just 15% — which
   matters enormously, because a sampler starting from all-`[MASK]` spends its early
   steps at rates 1.2's model never trained on.
2. Iterative unmasking becomes a **principled ancestral sampler** for a defined
   forward process, not a heuristic we invented by poking a scorer.

And it is dramatically cheaper: we will match 1.2's sample quality in **8** network
evaluations instead of 112.

**How to use this notebook:** implement the reps, make the checkpoints pass.
Solutions at the bottom. Trains in ~30s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

# ---- the same heptad corpus as 1.1 and 1.2 ----
AA = 'ACDEFGHIKLMNPQRSTVWY'; N_AA = len(AA)
MASK = N_AA; VOCAB = N_AA + 1
STOI = {a: i for i, a in enumerate(AA)}; HYDRO = 'AILMFV'
FREQ = {'A':8.25,'R':5.53,'N':4.06,'D':5.46,'C':1.38,'Q':3.93,'E':6.72,'G':7.07,
        'H':2.27,'I':5.91,'L':9.66,'K':5.80,'M':2.41,'F':3.86,'P':4.74,'S':6.65,
        'T':5.36,'W':1.10,'Y':2.92,'V':6.87}
f = np.array([FREQ[a] for a in AA]); is_h = np.array([a in HYDRO for a in AA])
L, EPS, CORE = 28, 0.05, (0, 3)
p_core = np.where(is_h, f * (1 - EPS) / f[is_h].sum(), f * EPS / f[~is_h].sum())
p_non  = np.where(is_h, f * EPS / f[is_h].sum(),  f * (1 - EPS) / f[~is_h].sum())

def make_corpus(n):
    ph = rng.integers(0, 7, size=n); out = np.zeros((n, L), dtype=np.int64)
    for i in range(L):
        r = (i - ph) % 7; core = (r == CORE[0]) | (r == CORE[1])
        for m, p in ((core, p_core), (~core, p_non)):
            k = int(m.sum())
            if k: out[m, i] = rng.choice(N_AA, size=k, p=p)
    return out, ph

def decode(row): return ''.join(AA[i] for i in row)
def pattern(s): return ''.join('H' if c in HYDRO else '.' for c in s)
def heptad_score(s):
    return max(sum(1 for i, c in enumerate(s) if (((i - phi) % 7) in CORE) == (c in HYDRO))
               / len(s) for phi in range(7))

train, _ = make_corpus(8000)
Xt = torch.from_numpy(train)
real = [decode(r) for r in train[:500]]
hs = lambda S: float(np.mean([heptad_score(s) for s in S]))
REAL_HEPTAD = hs(real)

# what 1.2 achieved, for the comparison at the end
GIBBS_HEPTAD, GIBBS_NFE = 0.965, 4 * L      # 4 sweeps x 28 positions

print('corpus', train.shape, '| real heptad %.3f' % REAL_HEPTAD)
print('1.2 baseline: heptad %.3f using %d network evaluations' % (GIBBS_HEPTAD, GIBBS_NFE))

## Part 1 — the forward process is one line

Mask each token independently with probability `t`, where `t` is drawn fresh for each
sequence. Compare with 1.2:

```
1.2:   sel = rand(B, L) < 0.15          # always 15%
1.5:   sel = rand(B, L) < t[:, None]    # t ~ U(0, 1), per sequence
```

That is the entire structural difference between a masked language model and a masked
diffusion model. Everything else follows from it.

Note we also drop the 80/10/10 recipe. It existed in BERT to stop the model relying on
`[MASK]` never appearing at inference — but here `[MASK]` **is** the noise state and
appears constantly at sampling time, so there is no train/test mismatch to patch.

### Rep 1 — `mask_at_rate(x0, t)`
Mask each position of `x0` independently with probability `t[i]` for sequence `i`.
Return `(xt, m)` — the corrupted batch and the boolean mask of what was hidden.

In [ ]:
def mask_at_rate(x0, t):
    '''Corrupt x0 by masking each token with per-sequence probability t. -> (xt, m)'''
    # YOUR CODE HERE
    # hint: m = torch.rand(x0.shape) < t[:, None]; xt = x0.clone(); xt[m] = MASK
    raise NotImplementedError

# --- checkpoint ---
x0 = Xt[:2000]
xt, m = mask_at_rate(x0, torch.full((2000,), 0.3))
assert xt.shape == x0.shape and m.dtype == torch.bool
assert (xt[m] == MASK).all(), 'masked positions must hold the MASK token'
assert (xt[~m] == x0[~m]).all(), 'unmasked positions must be untouched'
assert abs(m.float().mean().item() - 0.3) < 0.03, 'about 30% should be masked'

lo, _ = mask_at_rate(x0, torch.zeros(2000))
hi, _ = mask_at_rate(x0, torch.ones(2000))
assert (lo == x0).all(), 't=0 leaves the sequence clean'
assert (hi == MASK).all(), 't=1 masks everything'
print('t=0 -> clean | t=0.3 -> %.1f%% masked | t=1 -> all [MASK] ✓'
      % (100 * m.float().mean()))

fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
for ax, tv in zip(axes, [0.0, 0.3, 0.6, 1.0]):
    xv, _ = mask_at_rate(Xt[:40], torch.full((40,), tv))
    ax.imshow((xv == MASK).numpy(), aspect='auto', interpolation='nearest',
              cmap=ListedColormap(['#eaeef4', BLUE]))
    ax.set_title('t = %.1f' % tv, fontsize=10); ax.set_xlabel('position')
axes[0].set_ylabel('sequence')
plt.tight_layout(); plt.show()
print('blue = [MASK]. The forward process just dissolves the sequence into mask tokens.')

## Part 2 — training

Identical to 1.2's masked loss, except `t` is random and we also feed `t` to the
network so it knows how corrupted its input is (exactly as the DDPM net took `t`).

**A note on the loss weighting.** The principled ELBO for absorbing-state diffusion
carries a `1/t` factor on each masked position — low-`t` steps are rare but
informative, so they get upweighted. It is unbiased but high-variance. We use the
plain unweighted average here for stability and simplicity; when I measured both on
this problem they produced samples of equivalent quality (heptad 0.97 vs 0.98). Worth
knowing the weight exists, and worth not pretending it changed anything here.

In [ ]:
class MHA(nn.Module):
    def __init__(self, d, nh):
        super().__init__(); self.nh, self.dh = nh, d // nh
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
    def forward(self, x):
        B, T, Cc = x.shape
        q, k, v = self.qkv(x).split(Cc, 2)
        sp = lambda z: z.view(B, T, self.nh, self.dh).transpose(1, 2)
        a = F.softmax(sp(q) @ sp(k).transpose(-2, -1) / math.sqrt(self.dh), -1)  # bidirectional
        return self.proj((a @ sp(v)).transpose(1, 2).contiguous().view(B, T, Cc))

class Block(nn.Module):
    def __init__(self, d, nh):
        super().__init__(); self.l1, self.l2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.at = MHA(d, nh)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        x = x + self.at(self.l1(x)); return x + self.mlp(self.l2(x))

class MaskedDiffusion(nn.Module):
    '''1.2's bidirectional transformer, now conditioned on the corruption level t.'''
    def __init__(self, d=64, nh=4, nl=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(VOCAB, d), nn.Embedding(L, d)
        self.tmlp = nn.Sequential(nn.Linear(1, d), nn.SiLU(), nn.Linear(d, d))
        self.blocks = nn.ModuleList([Block(d, nh) for _ in range(nl)])
        self.lnf, self.head = nn.LayerNorm(d), nn.Linear(d, VOCAB)
    def forward(self, idx, t):
        h = (self.tok(idx) + self.pos(torch.arange(idx.size(1), device=idx.device))
             + self.tmlp(t[:, None])[:, None, :])
        for b in self.blocks:
            h = b(h)
        return self.head(self.lnf(h))

model = MaskedDiffusion()
print('parameters:', sum(p.numel() for p in model.parameters()))

### Rep 2 — `md_loss(model, x0)`
Draw `t ~ U(0, 1)` per sequence, corrupt with `mask_at_rate`, and return the
cross-entropy at the masked positions only. Return `None` if nothing got masked.

In [ ]:
def md_loss(model, x0):
    '''Masked-diffusion training loss: predict the clean tokens under a random mask rate.'''
    # YOUR CODE HERE
    # hint: t = torch.rand(len(x0)); xt, m = mask_at_rate(x0, t)
    #       logits = model(xt, t); score only logits[m] against x0[m]
    raise NotImplementedError

# --- checkpoint ---
l = md_loss(model, Xt[:256])
assert l.dim() == 0 and l.requires_grad, 'need a differentiable scalar'
assert 2.5 < l.item() < 3.6, 'an untrained net should be near ln(21) = 3.04'
print('untrained loss %.3f  (ln(21) = %.3f, i.e. no knowledge yet) ✓' % (l.item(), math.log(VOCAB)))

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
hist, t0 = [], time.time()
for step in range(1, 1501):
    ix = torch.randint(0, len(Xt), (128,))
    loss = md_loss(model, Xt[ix])
    if loss is None:
        continue
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0 or step == 1:
        hist.append((step, loss.item()))
        if step % 500 == 0 or step == 1:
            print('step %4d  loss %.4f' % (step, loss.item()))
print('trained in %.1fs on CPU' % (time.time() - t0))

h = np.array(hist)
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(h[:, 0], h[:, 1], color=BLUE, lw=2)
ax.axhline(math.log(VOCAB), color=INK, ls=':', lw=1.5)
ax.text(1480, math.log(VOCAB), 'know nothing', ha='right', va='bottom', fontsize=9, color=INK)
ax.set_xlabel('step'); ax.set_ylabel('cross-entropy on masked tokens')
ax.set_title('one model, every corruption level'); ax.grid(alpha=.15)
plt.show()
print('The loss stays high because it averages over ALL t — including t near 1, where')
print('almost everything is masked and the phase is genuinely unknowable.')

## Part 3 — sampling by unmasking

Start from a fully masked sequence and walk `t` from `1` down to `0`. At each step:

1. run the model to get a distribution for **every** masked position,
2. sample a candidate token at each,
3. **commit** only some of them — enough to bring the mask count down to the next
   step's level — and leave the rest masked for later.

Step 3 is the whole art. Which ones do you commit?

- **random** — the faithful ancestral sampler for the absorbing process: reveal each
  masked token with probability `(t - s) / t`, independent of what the model thinks.
- **confidence** — commit the tokens the model is most sure about, and let the
  uncertain ones benefit from more context later. This is the MaskGIT trick.

Confidence-first is not the mathematically faithful sampler, but it is strikingly
better in practice, for the same reason 1.2's one-shot decoding failed: committing an
uncertain token early forces every later token to live with a bad guess.

### Rep 3 — `predict_tokens(model, x, t, temperature=1.0)`
One forward pass. Return `(samp, conf)`: a sampled token for every position, and the
probability the model assigned to the token it sampled there. Never emit `[MASK]`.

In [ ]:
@torch.no_grad()
def predict_tokens(model, x, t, temperature=1.0):
    '''-> (sampled tokens [n, L], confidence of each sampled token [n, L]).'''
    # YOUR CODE HERE
    # hint: logits = model(x, torch.full((len(x),), float(t))) / temperature
    #       logits[:, :, MASK] = float('-inf'); probs = softmax(logits, -1)
    #       samp = torch.multinomial(probs.reshape(-1, VOCAB), 1).reshape(x.shape)
    #       conf = probs.gather(-1, samp[..., None]).squeeze(-1)
    raise NotImplementedError

# --- checkpoint ---
xall = torch.full((16, L), MASK, dtype=torch.long)
samp, conf = predict_tokens(model, xall, 1.0)
assert samp.shape == (16, L) and conf.shape == (16, L)
assert (samp != MASK).all(), '[MASK] must never be emitted'
assert ((conf >= 0) & (conf <= 1)).all(), 'confidences are probabilities'
print('predict_tokens ✓  mean confidence from a blank sequence: %.3f' % conf.mean())

### Rep 4 — `md_sample(model, n, steps=16, strategy='confidence')`
The sampling loop. Walk `t` from 1 to 0 in `steps` increments. Each iteration, call
`predict_tokens`, then commit tokens according to `strategy`, and make sure nothing is
left masked at the end.

Because every sequence starts fully masked and follows the same schedule, they all
have the **same** number of masked positions at every step — so the confidence
strategy can use a single batched `torch.topk` rather than a per-row loop.

In [ ]:
@torch.no_grad()
def md_sample(model, n, steps=16, strategy='confidence', temperature=1.0):
    '''Generate n sequences by iterative unmasking. Returns a list of strings.'''
    # YOUR CODE HERE
    # hint:
    #   x = torch.full((n, L), MASK, dtype=torch.long); ts = torch.linspace(1, 0, steps + 1)
    #   for i in range(steps):
    #       t, s_next = ts[i], ts[i + 1];  masked = x == MASK
    #       samp, conf = predict_tokens(model, x, t, temperature)
    #       conf = conf.masked_fill(~masked, -1.0)        # never re-commit a fixed token
    #       'random':     keep = (torch.rand(n, L) < (t - s_next) / t) & masked
    #       'confidence': k = int(masked[0].sum()) - round(s_next * L)  (same for all rows)
    #                     idx = torch.topk(conf, k, dim=1).indices; scatter samp at idx
    #   finally: commit anything still masked
    raise NotImplementedError

# --- checkpoint ---
for strat in ('random', 'confidence'):
    out = md_sample(model, 200, steps=8, strategy=strat)
    assert len(out) == 200 and all(len(s) == L for s in out), 'wrong shape'
    assert all(c in AA for s in out for c in s), 'no MASK may survive to the output'
    print('%-11s 8 steps -> heptad %.3f' % (strat, hs(out)))
print('real data        -> heptad %.3f' % REAL_HEPTAD)
print('\nConfidence-first wins, and it costs exactly the same 8 forward passes ✓')

## Part 4 — the payoff: quality per network evaluation

Now put it against 1.2. That notebook's Gibbs sampler resampled one position at a
time for 4 sweeps — `4 x 28 = 112` forward passes — and scored `0.965`.

Here, each unmasking step is **one** forward pass that updates many positions at once.
So `steps` is literally the NFE, and we can plot the same axis we used in 1.4.

In [ ]:
STEPS = [2, 4, 8, 16, 28]
res = {s: [hs(md_sample(model, 400, steps=k, strategy=s)) for k in STEPS]
       for s in ('random', 'confidence')}

print(' steps   random   confidence')
for i, k in enumerate(STEPS):
    print('%5d    %.3f     %.3f' % (k, res['random'][i], res['confidence'][i]))
print('\n1.2 Gibbs: %.3f at %d NFE' % (GIBBS_HEPTAD, GIBBS_NFE))

fig, ax = plt.subplots(figsize=(5.8, 3.6))
ax.plot(STEPS, res['confidence'], '-o', color=BLUE, lw=2, ms=6, label='confidence-first')
ax.plot(STEPS, res['random'], '-o', color=GREEN, lw=2, ms=6, label='random order')
ax.axhline(REAL_HEPTAD, color=INK, ls='--', lw=1.5)
ax.text(28, REAL_HEPTAD, 'real data', ha='right', va='bottom', fontsize=9, color=INK)
ax.plot([GIBBS_NFE], [GIBBS_HEPTAD], marker='*', ms=15, color=INK, ls='none')
ax.annotate("1.2's Gibbs\n(112 NFE)", (GIBBS_NFE, GIBBS_HEPTAD), fontsize=8, color=INK,
            xytext=(-6, -30), textcoords='offset points', ha='right')
ax.set_xscale('log'); ax.set_xticks(STEPS + [GIBBS_NFE])
ax.set_xticklabels(STEPS + [GIBBS_NFE])
ax.set_xlabel('network evaluations per sample'); ax.set_ylabel('heptad score')
ax.set_title('masked diffusion reaches 1.2 quality ~14x cheaper')
ax.grid(alpha=.15); ax.legend(frameon=False, fontsize=9, loc='lower right')
plt.show()

### The honest caveat, again

Look at the confidence curve past 16 steps: it climbs **above** the real data's score.
Same over-typicality we saw with Gibbs in 1.2 — and here we know exactly why. Always
committing the model's most confident tokens is a greedy, low-temperature move; it
walks the reward-vs-diversity frontier from 0.2 toward the "too regular" end.

So the two strategies are not simply better and worse:

- **random** is the faithful sampler — it targets the model's actual distribution, and
  lands nearer the data's own statistics.
- **confidence** produces cleaner-looking sequences faster, at the cost of sampling a
  subtly different (over-typical) distribution.

If your goal is "sequences that score well on my metric", take confidence. If your
goal is "an honest sample from the learned distribution", that convenience is a bias,
and `temperature` is how you trade between them.

## Part 5 — conditional generation comes free

Because the reverse process only ever fills in `[MASK]`, conditioning is trivial:
**start from a sequence that already has your motif in it.** Those positions are not
masked, so the sampler never touches them, and every generated token is conditioned on
them from step one.

This is the same motif-scaffolding task as 1.2 — but now it is the model's native
sampling procedure rather than a special-cased Gibbs loop, and it takes ~8 forward
passes instead of ~90. This is exactly how DPLM and friends do scaffolded design.

### Rep 5 — `md_infill(model, scaffold, n, steps=8, strategy='confidence')`
`scaffold` uses `'?'` for free positions. Build the starting tensor with the known
residues fixed and `'?'` masked, then run the same unmasking loop — never committing
anything to a pinned position.

In [ ]:
@torch.no_grad()
def md_infill(model, scaffold, n, steps=8, strategy='confidence', temperature=1.0):
    '''Fill the '?' positions of a scaffold by iterative unmasking. -> n strings.'''
    # YOUR CODE HERE
    # hint: base = [MASK if c == '?' else STOI[c] for c in scaffold]
    #       x = torch.tensor(base).repeat(n, 1)
    #       then the SAME loop as md_sample — since pinned positions are not MASK,
    #       `masked = x == MASK` already excludes them automatically
    raise NotImplementedError

# --- checkpoint ---
scaf = list('?' * L)
for i in (0, 3, 7, 10):
    scaf[i] = 'L'
scaf = ''.join(scaf)

outs = md_infill(model, scaf, 200, steps=8)
assert len(outs) == 200 and all(len(o) == L for o in outs)
for o in outs[:20]:
    assert all(o[i] == 'L' for i in (0, 3, 7, 10)), 'pinned residues must survive'
score = hs(outs)
assert score > 0.85, 'infills should continue the register implied by the anchors'
print('scaffold:', scaf)
for o in outs[:5]:
    print('  ', o, pattern(o), 'heptad %.2f' % heptad_score(o))
print('\nmean heptad %.3f over 200 designs, 8 forward passes each ✓' % score)

## Reflection — what just transferred

- **Masked diffusion = the 1.2 objective + the 1.3 recipe.** Randomize the corruption
  level and condition on it, and a scorer becomes a generator with an ELBO. The code
  change was one line; the conceptual change is that there is now a defined forward
  process whose reverse you are learning.
- **`[MASK]` is the discrete analogue of Gaussian noise**, and "absorbing state" is why
  the reverse process only has to unmask — a far easier job than general denoising.
- **Decoding order is a real design decision.** Random is faithful; confidence-first is
  cheaper and cleaner but biased toward typicality. Same tradeoff as 0.2's temperature,
  now dressed as a scheduling choice.
- **Cost fell out of the architecture.** One forward pass updates many positions, so
  quality-per-NFE improved ~14x over Gibbs. Compare 1.4: same lesson, different space.
- **Conditioning is free in an absorbing process.** Pin the tokens you know, mask the
  rest. Motif scaffolding, inpainting, and inverse folding are all this one idea.
- **This is DPLM.** Scale the transformer, train on real proteins, and the method you
  just wrote is the current state of the art for discrete protein generation.

**Next rep:** `1.6 SE(3) frame diffusion` — the last generative core. Take diffusion
off the flat plane and onto the manifold of rigid motions, so we can generate protein
*backbones* (the per-residue frames you built in 0.1) rather than sequences. This is
the simplified idea behind RFdiffusion.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def mask_at_rate(x0, t):
    m = torch.rand(x0.shape) < t[:, None]
    xt = x0.clone()
    xt[m] = MASK
    return xt, m

def md_loss(model, x0):
    t = torch.rand(len(x0))
    xt, m = mask_at_rate(x0, t)
    if m.sum() == 0:
        return None
    return F.cross_entropy(model(xt, t)[m], x0[m])

@torch.no_grad()
def predict_tokens(model, x, t, temperature=1.0):
    logits = model(x, torch.full((len(x),), float(t))) / temperature
    logits[:, :, MASK] = float('-inf')                    # never emit the noise token
    probs = F.softmax(logits, dim=-1)
    samp = torch.multinomial(probs.reshape(-1, VOCAB), 1).reshape(x.shape)
    conf = probs.gather(-1, samp[..., None]).squeeze(-1)
    return samp, conf

def _unmask_loop(model, x, steps, strategy, temperature):
    '''Shared engine for md_sample and md_infill: only ever fills [MASK] slots.'''
    n = len(x)
    ts = torch.linspace(1.0, 0.0, steps + 1)
    n_free_start = int((x[0] == MASK).sum())
    for i in range(steps):
        t, s_next = float(ts[i]), float(ts[i + 1])
        masked = x == MASK
        if not masked.any():
            break
        samp, conf = predict_tokens(model, x, t, temperature)
        conf = conf.masked_fill(~masked, -1.0)            # only consider masked slots
        if strategy == 'random':
            p = (t - s_next) / max(t, 1e-9)
            keep = (torch.rand(n, L) < p) & masked
            x[keep] = samp[keep]
        else:
            target = int(round(s_next * n_free_start))    # how many should remain masked
            k = int(masked[0].sum()) - target
            if k > 0:
                idx = torch.topk(conf, k, dim=1).indices
                x.scatter_(1, idx, samp.gather(1, idx))
    rest = x == MASK                                       # commit any stragglers
    if rest.any():
        samp, _ = predict_tokens(model, x, 0.0, temperature)
        x[rest] = samp[rest]
    return [decode(row) for row in x.numpy()]

@torch.no_grad()
def md_sample(model, n, steps=16, strategy='confidence', temperature=1.0):
    x = torch.full((n, L), MASK, dtype=torch.long)
    return _unmask_loop(model, x, steps, strategy, temperature)

@torch.no_grad()
def md_infill(model, scaffold, n, steps=8, strategy='confidence', temperature=1.0):
    base = [MASK if c == '?' else STOI[c] for c in scaffold]
    x = torch.tensor(base, dtype=torch.long).repeat(n, 1)
    return _unmask_loop(model, x, steps, strategy, temperature)

print('reference solutions loaded — re-run the checkpoint cells above')